# G11project · Colab Pro正式实验

本Notebook使用Colab的NVIDIA GPU完成任务030–032、024–029。训练结果写入Google Drive，计算写入`/content`临时盘。Colab出现交互验证时请正常完成；不要使用Keep Alive、自动点击或规避运行时限制的脚本。Runtime被回收后，从头执行环境单元格，再重新执行同一正式训练单元格即可恢复。

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

In [ ]:
import os
import subprocess
from pathlib import Path

REPOSITORY_URL = "https://github.com/yangyu-rgb/G11project.git"
GIT_REF = "main"
REPOSITORY = Path("/content/G11project")
PERSISTENT_ROOT = Path("/content/drive/MyDrive/G11project-formal")
WORK_ROOT = Path("/content/g11-work")

if not (REPOSITORY / ".git").is_dir():
    subprocess.run(["git", "clone", REPOSITORY_URL, str(REPOSITORY)], check=True)
subprocess.run(["git", "-C", str(REPOSITORY), "fetch", "origin"], check=True)
subprocess.run(["git", "-C", str(REPOSITORY), "checkout", GIT_REF], check=True)
subprocess.run(["git", "-C", str(REPOSITORY), "pull", "--ff-only", "origin", GIT_REF], check=True)
PERSISTENT_ROOT.mkdir(parents=True, exist_ok=True)
WORK_ROOT.mkdir(parents=True, exist_ok=True)
os.chdir(REPOSITORY)
print("Commit:", subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip())

In [ ]:
subprocess.run(["apt-get", "update", "-qq"], check=True)
subprocess.run(["apt-get", "install", "-y", "-qq", "sumo", "sumo-tools"], check=True)
subprocess.run(["python", "-m", "pip", "install", "-q", "-e", "./BackEnd[dev]"], check=True)
os.environ["SUMO_HOME"] = "/usr/share/sumo"

In [ ]:
import shutil
import torch

assert torch.cuda.is_available(), "当前不是GPU Runtime，请在Runtime设置中选择GPU"
assert shutil.which("sumo"), "SUMO安装失败"
free_gib = shutil.disk_usage(PERSISTENT_ROOT).free / 1024**3
print("GPU:", torch.cuda.get_device_name(0))
print("PyTorch:", torch.__version__, "CUDA:", torch.version.cuda)
print(f"持久存储可用空间: {free_gib:.1f} GiB")
subprocess.run(["python", "BackEnd/scripts/verify_sumo.py"], check=True)
subprocess.run(
    [
        "python",
        "BackEnd/scripts/run_formal_pipeline.py",
        "--persistent-root",
        str(PERSISTENT_ROOT),
        "--work-root",
        str(WORK_ROOT),
        "--status",
    ],
    check=True,
)

## 首次运行：独立Smoke
Smoke使用另一个Drive目录，不会污染formal结果。它验证2配置/2种子缩小矩阵、SUMO、CUDA、模型保存和恢复入口。

In [ ]:
import json

SMOKE_ROOT = PERSISTENT_ROOT.parent / "G11project-smoke"
while True:
    status = subprocess.run(
        [
            "python",
            "BackEnd/scripts/run_formal_pipeline.py",
            "--persistent-root",
            str(SMOKE_ROOT),
            "--work-root",
            "/content/g11-smoke-work",
            "--smoke",
            "--allow-cpu",
            "--status",
        ],
        capture_output=True,
        text=True,
        check=True,
    )
    if json.loads(status.stdout)["next_stage"] is None:
        break
    result = subprocess.run(
        [
            "python",
            "BackEnd/scripts/run_formal_pipeline.py",
            "--stage",
            "next",
            "--persistent-root",
            str(SMOKE_ROOT),
            "--work-root",
            "/content/g11-smoke-work",
            "--time-budget-minutes",
            "45",
            "--smoke",
        ]
    )
    assert result.returncode in (0, 75), result.returncode
    if result.returncode == 75:
        break

## 正式运行
每次执行推进当前阶段，最多运行9小时。返回码75代表正常安全暂停，不是失败。重新连接Runtime后再次执行前面的安装/挂载单元格和本单元格即可。不要同时开启两个Runtime写同一个`PERSISTENT_ROOT`。

In [ ]:
result = subprocess.run(
    [
        "python",
        "BackEnd/scripts/run_formal_pipeline.py",
        "--stage",
        "next",
        "--persistent-root",
        str(PERSISTENT_ROOT),
        "--work-root",
        str(WORK_ROOT),
        "--time-budget-minutes",
        "540",
    ]
)
assert result.returncode in (0, 75), result.returncode

In [ ]:
subprocess.run(
    [
        "python",
        "BackEnd/scripts/run_formal_pipeline.py",
        "--persistent-root",
        str(PERSISTENT_ROOT),
        "--work-root",
        str(WORK_ROOT),
        "--status",
    ],
    check=True,
)